[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/choROPeNt/FFTjax/blob/main/notebooks/lin-elastic_mixed-BC.ipynb)

# Two-Phase Composite RVE — Mixed Strain/Stress Boundary Conditions

A more realistic walkthrough of FFTjax's mechanical solver: instead of prescribing the full
macroscopic strain tensor (as in [`lin-elastic_strain.ipynb`](./lin-elastic_strain.ipynb)), this
notebook prescribes a **mixed** boundary condition — displacement-controlled tension along x with
free (traction-free) lateral surfaces along y and z — the condition an actual tensile-test
specimen is under (axial extension imposed at the grips, lateral surfaces free to contract via
Poisson's effect) — rather than the artificially-constrained uniaxial *strain* condition used
there, which locks the lateral strains to zero. The point of this setup is to actually measure the
composite's effective Poisson's ratios, which the fully-constrained case cannot do.

This uses `solvers.mechanical.displacement_nw_cg.ddisp_nw_cg`, the displacement-based solver --
required here because the strain-based solver's cheaper mixed-BC variant
(`dstrain_nw_cg_mixed`) is only valid for **homogeneous** materials (the mixed-BC CG operator loses
the symmetry a one-shot solve needs otherwise); our fibre/matrix composite is heterogeneous, so this
is the correct choice.

## Setup

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Running on Colab — installing FFTjax...")
    %pip install -q git+https://github.com/choROPeNt/FFTjax.git
else:
    print("Running locally — using the local src/ checkout.")

In [ ]:
import sys
sys.path.insert(0, "../src")
import os

import utils.precision  # side effect: configures JAX (X64 off on TPU, no GPU prealloc)
import jax
import jax.numpy as jnp
import numpy as np

from generation.rve import make_square_composite_rve
from operators.green import build_freq_grid
from mat_models.elastic import LinearElasticIsotropic, assemble_C_field
from solvers.mechanical.displacement_nw_cg import ddisp_nw_cg
from post.fields import field_to_grid, von_mises, compute_displacement
from utils.io.xdmf_writer import IncrementalWriter
from post.fields import to_voigt

import matplotlib.pyplot as plt

print("JAX backend:", jax.default_backend())
print("Devices:", jax.devices())
print("X64 enabled:", utils.precision.X64_ENABLED,
      "-> dtype:", jnp.zeros(1).dtype)

## Generate the composite RVE

`generation.rve.make_square_composite_rve` builds a square-packed 2-fibre RVE: a matrix phase with
circular fibre cross-sections arranged on a square lattice, extruded along Z into a 3-D voxel grid.
Same geometry as `lin-elastic_strain.ipynb`, so the two notebooks are directly comparable.

In [ ]:
phase_np, N, n, L, phi_act = make_square_composite_rve(
    phi=0.5, r_fiber=0.005, dx=0.0002, N_min=32, nz=10,
)
Nv = int(np.prod(n))

print("grid n :", n)
print("domain L [mm]:", tuple(float(Li) for Li in L))
print("fiber volume fraction (actual):", phi_act)

In [ ]:
# Visualize the fiber cross-section in the XY plane (Z=0)
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(phase_np[:, :, 0].T, origin="lower", cmap="gray_r")
ax.set_title(f"Fiber cross-section (Vf={phi_act:.3f})")
ax.set_xlabel("x [voxel]")
ax.set_ylabel("y [voxel]")
plt.show()

## Materials and stiffness field

A glass fiber in an epoxy matrix — a common, high-contrast (~23x stiffness ratio) composite.

In [ ]:
matrix = LinearElasticIsotropic(E=3.0e3,  nu=0.35, name="epoxy matrix")
fiber  = LinearElasticIsotropic(E=70.0e3, nu=0.20, name="glass fiber")

phase = jnp.array(phase_np.reshape(-1))   # 0 = matrix, 1 = fiber
C_field = assemble_C_field([matrix, fiber], phase)

print(matrix)
print(fiber)

## Mixed boundary conditions and solve

Prescribe displacement-controlled tension along x and traction-free lateral surfaces along y, z --
the standard setup for measuring an effective Poisson's ratio in a virtual tensile test:

- `control[0][0] = 0` (strain-controlled): `eps_bar[0, 0] = 1e-3` sets the axial tension directly.
- `control[1][1] = control[2][2] = 1` (stress-controlled): `stress_goal` is zero there, i.e. free
  lateral surfaces -- the solver finds whatever lateral strain makes `sigma22 = sigma33 = 0`.
- Shear components stay strain-controlled at zero (no shear loading).

`ddisp_nw_cg` solves for the displacement fluctuation and the free macroscopic strain components
(`eps_bar_out`, i.e. the lateral contraction) jointly, with no reference-medium approximation (the
true heterogeneous `C_field` is used directly, preconditioned by its voxel average). The two
Poisson's ratios printed below follow directly from the solved lateral strains.

In [ ]:
L_mm = tuple(float(Li) for Li in L)
dx = tuple(Li / ni for Li, ni in zip(L_mm, n))
xi_flat = build_freq_grid(n, L_mm)

eps_bar = jnp.array([
    [1.0e-3, 0.0, 0.0],
    [0.0,  0.0, 0.0],
    [0.0,  0.0, 0.0],
]) # xx and shear entries (control == 0, strain-controlled) are used
control = (
    (0, 0, 0),
    (0, 1, 0),
    (0, 0, 1),
)  # xx strain-controlled (tension); yy, zz stress-controlled (free lateral surfaces); shear strain-controlled = 0
stress_goal = jnp.array([
    [0.0, 0.0, 0.0],
    [0.0,  0.0, 0.0],
    [0.0,  0.0, 0.0],
]) # only yy, zz entries (control == 1) are used -- both 0, i.e. free lateral surfaces

eps, sigma, delta, eps_bar_out, converged = ddisp_nw_cg(
    n, C_field, xi_flat, eps_bar, control, stress_goal, toler_lin=1e-6, maxiter=2000,
)

print("converged     :", bool(converged))
print("eps_bar (solved):")
print(np.array(eps_bar_out))
print()
print("sigma11 (avg) :", float(jnp.mean(sigma[0, 0])), "MPa")
print("sigma22 (avg) :", float(jnp.mean(sigma[1, 1])), "MPa  (target: 0, free surface)")
print("sigma33 (avg) :", float(jnp.mean(sigma[2, 2])), "MPa  (target: 0, free surface)")
print()
print("effective nu_xy = -eps22/eps11 :", float(-eps_bar_out[1, 1] / eps_bar_out[0, 0]))
print("effective nu_xz = -eps33/eps11 :", float(-eps_bar_out[2, 2] / eps_bar_out[0, 0]))

The free lateral surfaces let the composite contract under axial load — exactly what a real
tensile specimen does (Poisson's effect) — which the constrained uniaxial-*strain* case in
`lin-elastic_strain.ipynb` cannot show, since it locks `eps22 = eps33 = 0` by construction. The two
effective Poisson ratios above also differ from each other: the square-packed fibres (aligned along
Z) make this composite's in-plane (xy) and through-thickness (xz) responses genuinely anisotropic,
not the single-value isotropic Poisson's ratio either constituent has on its own.

## Post-processing

Now we can visualize the results and also export them as a `.xdmf`/`.h5` pair for further
post-processing in ParaView or other visualization software, via FFTjax's `IncrementalWriter`
(the project-wide standard for field-data output). Every field here -- displacement, strain,
stress, phase -- is evaluated on the same voxel grid, so all of them are written voxel-centered
(`Center="Cell"`); there's no FEM-style node/cell split in this spectral scheme, so there's nothing
to gain from writing displacement at a different resolution than everything else.

In [ ]:
eps_grid   = field_to_grid(eps, n)
sigma_grid = field_to_grid(sigma, n)
u_grid     = compute_displacement(eps, eps_bar_out, xi_flat, n, dx)
vm_grid    = von_mises(sigma_grid)

eps_voigt   = to_voigt(eps_grid).astype(np.float64)
sigma_voigt = to_voigt(sigma_grid).astype(np.float64)

In [ ]:
VOIGT_LABELS = ["x", "y", "z", "xy", "xz", "yz"]
extent = [0.0, n[0] * dx[0], 0.0, n[1] * dx[1]]  # physical [mm] extent, binned by voxel size dx

fig, axes = plt.subplots(3, 3, figsize=(10, 9))

im = axes[0, 0].imshow(phase_np[:, :, 0].T, origin="lower", cmap="gray_r", extent=extent)
axes[0, 0].set_title("Fiber phase")
fig.colorbar(im, ax=axes[0, 0])

im = axes[0, 1].imshow(u_grid[:, :, 0, 0].T, origin="lower", cmap="plasma", extent=extent)
axes[0, 1].set_title(r"Displacement $u_x$ [mm]")
fig.colorbar(im, ax=axes[0, 1], format="%.1e")

im = axes[0, 2].imshow(u_grid[:, :, 0, 1].T, origin="lower", cmap="plasma", extent=extent)
axes[0, 2].set_title(r"Displacement $u_y$ [mm]")
fig.colorbar(im, ax=axes[0, 2], format="%.1e")

for idx, i in enumerate([0, 1, 3]):
    eps_plot = eps_voigt[:, :, 0, i]
    label = rf"$\varepsilon_{{{VOIGT_LABELS[i]}}}$"
    if i == 3:  # shear component: report engineering shear strain gamma = 2*epsilon
        eps_plot = 2.0 * eps_plot
        label = rf"$\gamma_{{{VOIGT_LABELS[i]}}}$"

    im = axes[1, idx].imshow(eps_plot.T, origin="lower", cmap="plasma", extent=extent)
    axes[1, idx].set_title(f"Strain {label}")
    fig.colorbar(im, ax=axes[1, idx], format="%.1e")

    im = axes[2, idx].imshow(sigma_voigt[:, :, 0, i].T, origin="lower", cmap="plasma", extent=extent)
    axes[2, idx].set_title(rf"Stress $\sigma_{{{VOIGT_LABELS[i]}}}$ [MPa]")
    fig.colorbar(im, ax=axes[2, idx], format="%.1f")

for ax in axes.flat:
    ax.set_xlabel("x [mm]")
    ax.set_ylabel("y [mm]")

plt.tight_layout()
plt.show()

In [ ]:
output_dir = "../output"
os.makedirs(output_dir, exist_ok=True)

with IncrementalWriter(f"{output_dir}/composite_rve_mixed_bc", grid_shape=n, grid_spacing=dx) as w:
    w.write_increment(0, {
        "phase":        phase_np.astype(np.float64),
        "displacement": u_grid.astype(np.float64),
        "strain":       to_voigt(eps_grid).astype(np.float64),
        "stress":       to_voigt(sigma_grid).astype(np.float64),
        "von_mises":    vm_grid.astype(np.float64),
    }, time=0.0)

print(f"Wrote {output_dir}/composite_rve_mixed_bc.h5")
print(f"      {output_dir}/composite_rve_mixed_bc.xdmf")
print("Open the .xdmf in ParaView with the 'Xdmf3ReaderT' reader.")

## Next steps

- Compare against [`lin-elastic_strain.ipynb`](./lin-elastic_strain.ipynb)'s pure uniaxial-*strain*
  case on the same geometry/materials — same load direction, very different lateral response.
- For a **homogeneous** material, `solvers.mechanical.strain_nw_cg.dstrain_nw_cg_mixed` solves the
  same kind of mixed BC more cheaply, reusing the fixed reference-medium Green's operator instead of
  the true heterogeneous stiffness -- not valid here since it loses symmetry for heterogeneous
  materials.
- See `solvers.damage.anderson` and the [Phase-Field Fracture example](../documentation/examples/phase-field.md)
  for coupling this kind of mechanical solve to damage evolution.